# Normalized Noise Waveform Analysis

This notebook analyzes the repo's waveform scaling without changing the main training or evaluation pipeline.

Goals:
- define a normalized noise level relative to the generated waveform power,
- show a surface over `alpha` and normalized `sigma^2`, and
- show how `alpha` changes actual noise power when the noise is normalized to the waveform power.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from config import ExperimentConfig
from utils.data_utils.generator import RFMixtureGenerator, QPSKConfig, NoiseConfig, MixtureConfig

## Helper Functions

`normalized_sigma2` in this notebook means a *ratio* relative to the clean waveform power:

`noise_power = normalized_sigma2 * signal_power`

This is separate from the repo's raw `NoiseConfig.sigma2`, which is an absolute complex noise variance added at the IQ sample level.

In [ ]:
def generate_clean_mixture(alpha: float, seed: int, n_rx: int, n_symbols: int, sps: int, rolloff: float, span: int, phase_shift_deg: int):
    gen = RFMixtureGenerator(seed=seed)
    qpsk_cfg = QPSKConfig(
        n_symbols=n_symbols,
        samples_per_symbol=sps,
        rolloff=rolloff,
        rrc_span_symbols=span,
        normalize_power=True,
        num_channels=n_rx,
    )
    noise_cfg = NoiseConfig(enabled=False, snr_db=None, sigma2=None)
    mix_cfg = MixtureConfig(
        alpha=float(alpha),
        snr_db=None,
        n_rx=n_rx,
        random_phase=False,
        phase_shift_deg=phase_shift_deg,
        interference_phase_shift=0,
    )
    ex = gen.generate_mixture(qpsk_cfg, qpsk_cfg, noise_cfg, mix_cfg)
    signal = ex['mixture']
    return signal if signal.ndim == 1 else signal.reshape(-1)


def make_normalized_noise(signal: np.ndarray, normalized_sigma2: float, seed: int):
    rng = np.random.default_rng(seed)
    signal_power = float(np.mean(np.abs(signal) ** 2))
    noise_power = float(normalized_sigma2 * signal_power)

    if noise_power == 0.0:
        noise = np.zeros_like(signal)
        realized_noise_power = 0.0
        effective_snr_db = np.inf
    else:
        noise = (
            rng.standard_normal(signal.shape) + 1j * rng.standard_normal(signal.shape)
        ) / np.sqrt(2.0)
        noise *= np.sqrt(noise_power)
        realized_noise_power = float(np.mean(np.abs(noise) ** 2))
        effective_snr_db = 10.0 * np.log10(signal_power / noise_power)

    return {
        'signal_power': signal_power,
        'target_noise_power': noise_power,
        'realized_noise_power': realized_noise_power,
        'effective_snr_db': effective_snr_db,
    }


def build_metric_grids(alpha_values, normalized_sigma2_values, trials, n_rx, n_symbols, sps, rolloff, span, phase_shift_deg):
    signal_power_grid = np.zeros((len(alpha_values), len(normalized_sigma2_values)), dtype=float)
    target_noise_power_grid = np.zeros((len(alpha_values), len(normalized_sigma2_values)), dtype=float)
    realized_noise_power_grid = np.zeros((len(alpha_values), len(normalized_sigma2_values)), dtype=float)
    effective_snr_grid = np.zeros((len(alpha_values), len(normalized_sigma2_values)), dtype=float)

    for ai, alpha in enumerate(alpha_values):
        for si, normalized_sigma2 in enumerate(normalized_sigma2_values):
            signal_powers = []
            target_noise_powers = []
            realized_noise_powers = []
            snrs = []

            for trial in range(trials):
                clean = generate_clean_mixture(alpha, 1000 * ai + trial, n_rx, n_symbols, sps, rolloff, span, phase_shift_deg)
                stats = make_normalized_noise(clean, normalized_sigma2, 2000 * ai + 100 * si + trial)
                signal_powers.append(stats['signal_power'])
                target_noise_powers.append(stats['target_noise_power'])
                realized_noise_powers.append(stats['realized_noise_power'])
                snrs.append(stats['effective_snr_db'])

            signal_power_grid[ai, si] = float(np.mean(signal_powers))
            target_noise_power_grid[ai, si] = float(np.mean(target_noise_powers))
            realized_noise_power_grid[ai, si] = float(np.mean(realized_noise_powers))
            effective_snr_grid[ai, si] = float(np.mean(snrs))

    return signal_power_grid, target_noise_power_grid, realized_noise_power_grid, effective_snr_grid

## Analysis Parameters

The defaults below align with the repo's notebook-style single-channel experiments, but `analysis_n_rx` can be changed to `2` or `4` to inspect other receive-array settings.

In [ ]:
analysis_n_rx = 1
analysis_phase_shift_deg = ExperimentConfig.phase_shift_deg
analysis_n_symbols = 100
analysis_sps = ExperimentConfig.samples_per_symbol
analysis_rolloff = ExperimentConfig.rolloff
analysis_span = ExperimentConfig.rrc_span_symbols
analysis_trials = 8

alpha_values = np.round(np.arange(0.0, 2.0 + 0.1, 0.1), 1)
normalized_sigma2_values = np.array([0.0, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0], dtype=float)

In [ ]:
signal_power_grid, target_noise_power_grid, realized_noise_power_grid, effective_snr_grid = build_metric_grids(
    alpha_values=alpha_values,
    normalized_sigma2_values=normalized_sigma2_values,
    trials=analysis_trials,
    n_rx=analysis_n_rx,
    n_symbols=analysis_n_symbols,
    sps=analysis_sps,
    rolloff=analysis_rolloff,
    span=analysis_span,
    phase_shift_deg=analysis_phase_shift_deg,
)

## Surface Plot

This surface shows the *actual complex noise power* that gets added when noise is normalized to the clean waveform power. The x-axis is `alpha`, the y-axis is normalized `sigma^2` ratio, and the z-axis is the resulting noise power.

In [ ]:
norm_sigma2_grid, alpha_grid = np.meshgrid(normalized_sigma2_values, alpha_values)

fig = plt.figure(figsize=(10, 7), constrained_layout=True)
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(alpha_grid, norm_sigma2_grid, target_noise_power_grid, cmap='viridis', edgecolor='none')

ax.set_xlabel('alpha')
ax.set_ylabel('normalized sigma^2 ratio')
ax.set_zlabel('actual complex noise power')
ax.set_title('Noise Power After Normalizing To Clean Waveform Power')
fig.colorbar(surf, ax=ax, shrink=0.7, pad=0.1)
plt.show()

## 2D Alpha Plots At Different Normalized Noise Levels

The left plot shows how the clean mixture power increases with `alpha`. The right plot shows how the *actual* noise power increases with `alpha` when the requested noise level is defined as a normalized fraction of the waveform power.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].plot(alpha_values, signal_power_grid[:, 0], linewidth=2, label='clean mixture power')
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('clean mixture power')
axes[0].set_title('Clean Mixture Power vs Alpha')
axes[0].grid(True, linestyle='--', alpha=0.5)

for normalized_sigma2 in [0.01, 0.1, 0.5, 1.0, 2.0]:
    idx = int(np.where(np.isclose(normalized_sigma2_values, normalized_sigma2))[0][0])
    axes[1].plot(alpha_values, target_noise_power_grid[:, idx], linewidth=2, label=f'norm sigma^2={normalized_sigma2:g}')

axes[1].set_xlabel('alpha')
axes[1].set_ylabel('actual complex noise power')
axes[1].set_title('Alpha Effect At Different Normalized Noise Levels')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

plt.show()

## Optional Check: Effective SNR

When noise is defined as a normalized fraction of the waveform power, the effective sample-level SNR should stay approximately flat across `alpha` for a fixed normalized `sigma^2` ratio.

In [ ]:
plt.figure(figsize=(8, 5), constrained_layout=True)
for normalized_sigma2 in [0.01, 0.1, 0.5, 1.0, 2.0]:
    idx = int(np.where(np.isclose(normalized_sigma2_values, normalized_sigma2))[0][0])
    plt.plot(alpha_values, effective_snr_grid[:, idx], linewidth=2, label=f'norm sigma^2={normalized_sigma2:g}')

plt.xlabel('alpha')
plt.ylabel('effective sample-level SNR (dB)')
plt.title('Effective SNR vs Alpha Under Normalized Noise')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()